In [1]:
import logging
import pathlib
import threading
from pprint import pprint

from dotenv import load_dotenv

# ── pycram ─────────────────────────────────────────────────────────────────────
from pycram.motion_executor import simulated_robot

# ── Generative backend ─────────────────────────────────────────────────────────
from llmr import ExecutionLoop, ActionPipeline, TaskDecomposer
from llmr import load_pr2_apartment_world
from llmr.pipeline import ActionDispatcher

# Load API keys before any LLM call
_env = pathlib.Path().resolve().parent / ".env"
load_dotenv(_env, override=True)
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("generative_backend").setLevel(logging.DEBUG)

print("All imports OK")

/home/malineni/workingdir/cognitive_robot_abstract_machine/probabilistic_model/src/probabilistic_model/probabilistic_circuit/rx/probabilistic_circuit.py:506: SyntaxWarning: invalid escape sequence '\_'
  """
/home/malineni/workingdir/cognitive_robot_abstract_machine/probabilistic_model/src/probabilistic_model/probabilistic_model.py:260: SyntaxWarning: invalid escape sequence '\i'
  """


All imports OK


In [2]:
# Build world (parses URDFs fresh — semantic annotations preserved)
world, pr2, context = load_pr2_apartment_world()
print("World loaded")

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


World loaded


In [3]:
world.__dict__.keys()

dict_keys(['simulator_additional_properties', 'kinematic_structure', 'semantic_annotations', 'degrees_of_freedom', 'actuators', 'world_is_being_modified', 'name', '_id', '_model_manager', '_world_entity_hash_table', '_world_lock', 'state', '_forward_kinematic_manager', 'collision_manager', '_current_active_atomic_world_modification'])

In [4]:
len(world.semantic_annotations)

15

In [5]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from llmr import ExecutionLoop, RecoveryHandler

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [6]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


In [7]:
def reset_world():
    """Reset world to base state without restarting the kernel."""
    global world, pr2, context, _tf_publisher, _viz_publisher
    world, pr2, context = load_pr2_apartment_world()
    _tf_publisher = TFPublisher(_world=world, node=_ros_node)
    _viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
    print("World reset OK")


print("reset_world() ready — call it to restore the world between runs")

reset_world() ready — call it to restore the world between runs


In [8]:
pipeline = ActionPipeline.from_world(world)
print("ActionPipeline ready")
print(f"  manipulator      : {type(pipeline.world_context.manipulator).__name__}")
print(f"  registered types : {list(ActionDispatcher._registry.keys())}")

ActionPipeline ready
  manipulator      : ParallelGripper
  registered types : ['PickUpAction', 'PlaceAction']


In [9]:
pipeline.__dict__['world_context'].__dict__.keys()

dict_keys(['manipulator'])

In [10]:
handler = RecoveryHandler(world=world, max_retries=2)

loop = ExecutionLoop(
    world=world,
    pipeline=pipeline,
    context=context,
    robot_context=lambda: simulated_robot,
    decomposer=TaskDecomposer(),
    recovery_handler=handler
)
print("ExecutionLoop ready")

ExecutionLoop ready


In [11]:
results, session = loop.run([
# "fetch the milk and place it on the island_countertop",
    "Pick up the milk from the counter",
#     "pick up the breakfast_cereal from the counter",
# "Pick up the mug from the counter",
"Place the milk on the island_countertop",
# "pick up the breakfast_cereal from the counter",
#     "Place the breakfast_cereal on the island_countertop",
# "Place the milk on the island_countertop",
])

INFO:base.py::38 perform Performing action ParkArmsAction
INFO:base.py::38 perform Performing action MoveTorsoAction
INFO:base.py::38 perform Performing action NavigateAction
INFO:base.py::38 perform Performing action PickUpAction
INFO:base.py::38 perform Performing action ReachAction
INFO:base.py::38 perform Performing action MoveTorsoAction
ERROR:execution_loop.py::285 run Execution failed for 'place the milk on the island_countertop': <built-in function mtimes> returned a result with an exception set


In [ ]:
for r in results:
    status = "OK" if r.success else f"FAILED: {r.error}"
    actions = [type(a).__name__ for a in [*r.preconditions, r.action]] if r.action else []
    print(f"{r.instruction!r}  →  {status}")
    if actions:
        print(f"  actions: {actions}")

In [ ]:
##ORM
from sqlalchemy import select
from pycram.robot_plans import NavigateAction
from pycram.robot_plans.actions.base import ActionDescription
from pycram.orm.ormatic_interface import NavigateActionDAO, ActionDescriptionDAO

navigations = session.scalars(select(ActionDescriptionDAO)).all()
print(*navigations, sep="\n")

In [ ]:
pickupdao = navigations[3]

In [ ]:
allactions = session.scalars(select(ActionDescriptionDAO)).all()

In [ ]:
allactions[-2].__dict__

## Debuggs

In [ ]:
import json
from pprint import pprint

In [ ]:

pipe = ActionPipeline.from_world(world)

In [ ]:
slot_schema = pipe.classify_and_extract("pick up the milk from the table")
print(slot_schema.model_dump_json(indent=2))

In [ ]:
action = pipe.dispatch(schema=slot_schema)

In [ ]:
print(type(action))
print(action.__dict__.keys())

In [ ]:
pprint(action.__dict__['object_designator'])

In [ ]:
pprint(action.__dict__['arm'])

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, kw_only=True, order=True)
class Animal:
    name : str
    id : int

    def __post_init__(self):
        if self.id>10:
            raise ValueError("id must be less than 10")

In [ ]:
a = Animal(name="lion",id=2)

In [ ]:
set = {a}

In [12]:
# claude --resume "migrate-llmr-pycram-refactor"